# 3 - RandomForest

maintenant que tout est traité, on peut faire l'entrainement

In [1]:
import numpy as np
import pandas as pd
import sklearn as skl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, mean_absolute_error

In [2]:
# Usine N°1
DF_U1_Train = pd.read_csv("Data/csv/train_usine1.csv")
DF_U1_Test = pd.read_csv("Data/csv/test_usine1.csv")
DF_U1_Rul = pd.read_csv("Data/csv/rul_usine1.csv")

# Usine N°2
DF_U2_Train = pd.read_csv("Data/csv/train_usine2.csv")
DF_U2_Test = pd.read_csv("Data/csv/test_usine2.csv")
DF_U2_Rul = pd.read_csv("Data/csv/rul_usine2.csv")

In [3]:
# Features utiles : (determiné dans "Anlayse"
Features = ['cycle',
            'Température sortie LPC — T24',
            'Température sortie HPC — T30',
            'Température sortie LPT — T50',
            'Pression bypass — P15',
            'Pression sortie HPC — P30',
            'Vitesse fan — Nf',
            'Vitesse cœur — Nc',
            'Rapport pression moteur — EPR',
            'Pression statique HPC — Ps30',
            'Ratio carburant/pression — phi',
            'Vitesse corrigée fan — NRf',
            'Vitesse corrigée cœur — NRc',
            'Bypass Ratio — BPR',
            'Soutirage HPC — htBleed',
            'Débit refroid. HPT — W31',
            'Débit refroid. LPT — W32']

# répartitions
X_train_U1 = DF_U1_Train[Features]
y_train_U1 = DF_U1_Train["RUL"]

In [4]:
scaler = MinMaxScaler()
X_train_U1_scaled = scaler.fit_transform(X_train_U1)       # calcule min/max sur le train

# Sur le test : appliquer les MÊMES min/max, sans recalculer
X_test_U1_scaled = scaler.transform(DF_U1_Test[Features])

In [5]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_U1_scaled, y_train_U1)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [6]:
DF_result  = scaler.transform(DF_U1_Test.groupby('machine_uid').last()[Features])
# Prédictions
y_Pred_U1 = rf.predict(DF_result)

In [7]:
y_true_U1 = DF_U1_Rul['RUL']

R2 = r2_score(y_true_U1, y_Pred_U1)
MAE = mean_absolute_error(y_true_U1, y_Pred_U1)
MSE = mean_squared_error(y_true_U1, y_Pred_U1)
RMSE = np.sqrt(MSE)

print(" ---------- Resultat ----------")
print(f"R² : {R2:.4f}")
print(f"MAE (erreur moyenne): {MAE:.2f}")
print(f"MSE (erreur au carré) : {MSE:.2f}")
print(f"RMSE (erreur écart-type) : {RMSE:.2f}")

 ---------- Resultat ----------
R² : -0.9027
MAE (erreur moyenne): 45.53
MSE (erreur au carré) : 3273.17
RMSE (erreur écart-type) : 57.21


In [8]:
# Ordre des machines dans les prédictions
print(DF_U1_Test.groupby('machine_uid').last().index[:5])

# Ordre des machines dans le RUL vrai
print(DF_U1_Rul['machine_uid'][:5])

Index(['100_1_1', '100_1_2', '10_1_1', '10_1_2', '11_1_1'], dtype='object', name='machine_uid')
0    1_1_1
1    2_1_1
2    3_1_1
3    4_1_1
4    5_1_1
Name: machine_uid, dtype: object


Les Rul est triée numériquement alors que mon test est alphabétique 

In [9]:
# L'ordre des machine_uid dans tes prédictions
ordre = DF_U1_Test.groupby('machine_uid').last().index

# Réordonner le RUL vrai dans ce même ordre
y_true = DF_U1_Rul.set_index('machine_uid').loc[ordre, 'RUL']
y_true_U1 = DF_U1_Rul['RUL']

R2 = r2_score(y_true, y_Pred_U1)
MAE = mean_absolute_error(y_true, y_Pred_U1)
MSE = mean_squared_error(y_true, y_Pred_U1)
RMSE = np.sqrt(MSE)

print(" ---------- Resultat ----------")
print(f"R² : {R2:.4f}")
print(f"MAE (erreur moyenne): {MAE:.2f}")
print(f"MSE (erreur au carré) : {MSE:.2f}")
print(f"RMSE (erreur écart-type) : {RMSE:.2f}")

 ---------- Resultat ----------
R² : 0.8048
MAE (erreur moyenne): 13.64
MSE (erreur au carré) : 335.83
RMSE (erreur écart-type) : 18.33


In [10]:
# Score PHM08 asymétrique
def phm_score_sur(rul_true, rul_pred):
    d = rul_pred - rul_true
    s = np.where(d < 0, np.exp(-d / 13) - 1, 0)
    return float(np.sum(s))

def phm_score_dang(rul_true, rul_pred):
    d = rul_pred - rul_true
    s = np.where(d > 0, np.exp(d / 10) - 1, 0)
    return float(np.sum(s))

PHMS = phm_score_sur(y_true, y_Pred_U1)
PHMD = phm_score_dang(y_true, y_Pred_U1)
print(f"PHM08 Sur : {PHMS/100}")
print(f"PHM08 Dangereux : {PHMD/100}")

PHM08 Sur : 4.180011832313695
PHM08 Dangereux : 9.148694788096872


Le PHM08 sert a savoir si le modèle prédit en majorité + de cycle de vie ou moins.
PHMS est "Sur" car c'est lorsque le modèle prédit -
PHMD est "dangereux" car c'est lorsque le modèle prédit +, c'est a dire il surestime la durée de vie de la machine ce que l'on veut pas

Et en regardant les résultat le modèle prédit + de cycle que - de cycle

In [11]:
# Score PHM08 asymétrique
DF_PHMS =[]
def construire_table_erreurs(y_true, y_pred, machine_uid):
    d = y_pred - y_true
    df = pd.DataFrame({
        'y_true': y_true,
        'y_pred': y_pred,
        'd': d
    })
    return df

DF_PHMS = construire_table_erreurs(y_true, y_Pred_U1, y_true.index)
DF_PHMS

,y_true,y_pred,d
machine_uid,,,
100_1_1,20,16.45,-3.55
100_1_2,28,32.99,4.99
10_1_1,96,100.88,4.88
10_1_2,66,96.29,30.29
11_1_1,97,79.22,-17.78
...,...,...,...
98_1_2,17,20.46,3.46
99_1_1,117,120.32,3.32
99_1_2,8,11.08,3.08


In [12]:
DF_PHMS.sort_values('d', ascending=False).head(10)

,y_true,y_pred,d
machine_uid,,,
93_1_2,67,115.78,48.78
41_1_1,18,64.45,46.45
58_1_2,40,84.42,44.42
70_1_2,63,107.16,44.16
2_1_2,51,91.67,40.67
54_1_2,87,122.24,35.24
15_1_1,83,118.15,35.15
89_1_2,41,75.03,34.03
91_1_2,81,112.02,31.02


In [13]:
DF_PHMS.sort_values('d', ascending=True).head(10)

,y_true,y_pred,d
machine_uid,,,
83_1_2,145,90.98,-54.02
45_1_1,114,70.63,-43.37
30_1_1,115,73.71,-41.29
17_1_2,136,95.12,-40.88
74_1_1,126,86.34,-39.66
89_1_1,136,96.75,-39.25
51_1_1,114,75.32,-38.68
12_1_2,115,78.54,-36.46
93_1_1,85,48.84,-36.16


In [14]:
masque_risque = y_train_U1 <= 30
n_risque = masque_risque.sum()
n_total = len(y_train_U1)
proportion = n_risque / n_total
n_normaux = n_total - n_risque
print(f"Nombre de lignes a risque : {n_risque}") # donne le nombre de lignes du train où le RUL vrai est ≤ 30 
print(f"Nombre de lignes normaux : {n_normaux}") # le nombre de lignes du train "normaux"
print(f"Nombre de lignes totaux : {n_total}") # donne le nombre total de lignes du train
print(f"Pourcentage de critique dans le document : {proportion * 100}%") # dit quelle part ça représente dans tout le jeu d'entraînement.

ratio = (1 - proportion) / proportion
weights = np.where(masque_risque, 12, 1)
print(f"Facteur multiplicateur pour les 'A risque' : {ratio}")

Nombre de lignes a risque : 6200
Nombre de lignes normaux : 39151
Nombre de lignes totaux : 45351
Pourcentage de critique dans le document : 13.671142863442922%
Facteur multiplicateur pour les 'A risque' : 6.314677419354839


Ici, on voit que les zones a risques représente 13,67 % du dataset. Il faut donc ajouter un facteur de 6.3 a toute ces données afin de rééquilibrer

In [15]:
rf2 = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf2.fit(X_train_U1_scaled, y_train_U1, sample_weight=weights)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [16]:
y_Pred_U1b = rf2.predict(DF_result)
R2 = r2_score(y_true, y_Pred_U1b)
MAE = mean_absolute_error(y_true, y_Pred_U1b)
MSE = mean_squared_error(y_true, y_Pred_U1b)
RMSE = np.sqrt(MSE)
print(" ---------- Resultat ----------")
print(f"R² : {R2:.4f}")
print(f"MAE (erreur moyenne): {MAE:.2f}")
print(f"MSE (erreur au carré) : {MSE:.2f}")
print(f"RMSE (erreur écart-type) : {RMSE:.2f}")

 ---------- Resultat ----------
R² : 0.8101
MAE (erreur moyenne): 13.65
MSE (erreur au carré) : 326.70
RMSE (erreur écart-type) : 18.07


In [17]:
PHMS = phm_score_sur(y_true, y_Pred_U1b)
PHMD = phm_score_dang(y_true, y_Pred_U1b)
print(f"PHM08 Sur : {PHMS/100}")
print(f"PHM08 Dangereux : {PHMD/100}")

PHM08 Sur : 4.251233890675261
PHM08 Dangereux : 7.604191848488983
